# Radiate Analysis from Reactome stIds


This notebook runs a radiate PageRank analysis in the Reactome graph for a user-supplied list of Reactome stable identifiers (`stId`). It is intended as the first step in a two-notebook workflow.

**Before you run this notebook**

- Create a `.env` file with `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, and `NEO4J_DATABASE`.
- Place a CSV or TSV input file in `notebooks/input/` with a required `stId` column.
- Update `source_file`, `source_name`, and `source_desc` below to match your dataset.

**What this notebook writes**

- `output/Radiate_analysis_for_<source_name>.xlsx`: ranked forward and reverse radiate analysis results.

**Recommended workflow**

1. Run this notebook from top to bottom.
2. Review the exported workbook and mark rows of interest in the `select` column.
3. Save the edited workbook in `notebooks/input/` and then run `radiate_traces.ipynb`.


In [ ]:
import os
import warnings
from pathlib import Path

from lifelike_gds.network.trace_graph_nx import TraceGraphNx
from lifelike_gds.graph_sources.domain_config import REACTOME_TRACE_NODE_LABEL
from lifelike_gds.graph_sources.reactome_db import ReactomeDB
from lifelike_gds.graph_sources.reactome import Reactome
from lifelike_gds.network.radiate_trace import RadiateTrace

# Ignore warnings
warnings.filterwarnings('ignore')

## Settings


In [ ]:
import dotenv

# Load environment variables used to connect to Neo4j.
dotenv.load_dotenv()

required_env_vars = [
    'NEO4J_URI',
    'NEO4J_USERNAME',
    'NEO4J_PASSWORD',
    'NEO4J_DATABASE',
]
missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    raise ValueError(
        'Missing required environment variables: ' + ', '.join(missing_env_vars)
    )

# Keep notebook inputs and outputs relative to the notebook directory so the workflow
# is portable when shared with other users.
input_dir = Path('input')
output_dir = Path('output')
os.makedirs(output_dir, exist_ok=True)

print(f'Input directory: {Path(input_dir).resolve()}')
print(f'Output directory: {Path(output_dir).resolve()}')


In [ ]:
# Update these values when reusing the notebook for a different analysis.
source_file = 'als_related_6_genes.csv'
source_name = 'ALS_related_gene_rxns'
source_desc = 'Genes and reactions related to ALS'
source_path = input_dir / source_file

if not source_path.exists():
    raise FileNotFoundError(f'Input file not found: {source_path}')

print(f'Reading data from {source_path}')


## Read Input File and Collect Source stIds


In [ ]:
import pandas as pd

# `sep=None` lets pandas infer comma- vs tab-delimited text files.
df = pd.read_csv(source_path, sep=None, engine='python')
if 'stId' not in df.columns:
    raise ValueError("Input file must include a column named 'stId'.")

print(df.head())

st_ids = df['stId'].dropna().astype(str).drop_duplicates().tolist()
if not st_ids:
    raise ValueError("No Reactome stIds were found in the input file.")

print(f'Found {len(st_ids)} unique Reactome stIds in the input data')
print(st_ids)


## Helper Functions


In [ ]:
def _export_radiate_analysis(
    tracegraph: TraceGraphNx,
    source_name: str,
    source_description: str,
    source_nodes: list,
    exclude_sources_from_file: bool = False,
    rows_export: int = 2000,
):
    """Run radiate analysis and export the ranked results workbook."""
    tracegraph.graph = tracegraph.orig_graph.copy()
    tracegraph.set_node_set_from_db_nodes(
        source_nodes, source_name, source_description
    )
    outfile_name = f'Radiate_analysis_for_{source_name}.xlsx'
    tracegraph.export_pagerank_data(
        source_name,
        outfile_name,
        direction='both',
        num_nodes=rows_export,
        exclude_sources=exclude_sources_from_file,
    )
    return output_dir / outfile_name


def _get_database() -> ReactomeDB:
    return ReactomeDB(
        uri=os.getenv('NEO4J_URI'),
        username=os.getenv('NEO4J_USERNAME'),
        password=os.getenv('NEO4J_PASSWORD'),
        database=os.getenv('NEO4J_DATABASE'),
    )


## Resolve Reactome Nodes from the Input stIds


In [ ]:
database = _get_database()

# Look up input identifiers as Reactome trace nodes.
source_nodes = database.get_nodes_by_attr(
    attr_values=st_ids,
    attr_name='stId',
    node_label=REACTOME_TRACE_NODE_LABEL,
)
if not source_nodes:
    raise ValueError('No Reactome nodes were found for the provided stIds.')

print(f'Found {len(source_nodes)} source nodes in Reactome matching the input stIds')


## Load the Reactome Graph into Memory


In [ ]:
tracegraph = RadiateTrace(Reactome(database))

# Write notebook outputs to the shared `notebooks/output/` directory.
tracegraph.datadir = output_dir

# Build the in-memory NetworkX graph used by the radiate algorithm.
tracegraph.init_default_graph()


In [ ]:
analysis_file = _export_radiate_analysis(
    tracegraph,
    source_name,
    source_desc,
    source_nodes,
)
print(f'Radiate analysis exported to {analysis_file.resolve()}')


## Next Step

Open the exported workbook, review the `pageranks` and `reverse pageranks` sheets, and mark rows to keep by setting `select = 1`. Save that edited workbook into `notebooks/input/` before running `radiate_traces.ipynb`.
